# CARLA Python API - Project 5: Cooperative Roadside Assistant (V2X / I2V)

This notebook contains exactly 3 sections:
1. **Infrastructure Sensing & World Transformation** (Theory, camera setups, and 2D-to-3D projection layout)
2. **V2X MQTT Server Architecture & Message Serialization** (Live MQTT client setup, network delay emulation, and protocol payloads)
3. **End-to-End System Integration & Occlusion Scenarios** (The runnable project execution with full evaluation metrics)

## CARLA docs
- Main docs: https://carla.readthedocs.io/en/latest/
- Sensors reference: https://carla.readthedocs.io/en/latest/ref_sensors/
- Python API: https://carla.readthedocs.io/en/latest/python_api/

In [1]:
import carla
import time
import random
import cv2
import queue
import threading
import json
import math
import numpy as np
from datetime import datetime

# Initialize client and connect to the CARLA server daemon
client = carla.Client("localhost", 2000)
client.set_timeout(10.0)
world = client.get_world()
spectator = world.get_spectator()
blueprint_library = world.get_blueprint_library()

In [2]:
def move_spectator_to(transform, spectator, distance=12.0, z=6.0, pitch=-25.0):
    """Utility to orient the editor spectator view behind an active actor."""
    back = transform.location - transform.get_forward_vector() * distance
    loc = carla.Location(back.x, back.y, back.z + z)
    rot = carla.Rotation(pitch=pitch, yaw=transform.rotation.yaw, roll=0.0)
    spectator.set_transform(carla.Transform(loc, rot))

def safe_destroy(actors):
    """Safely removes lists of actors under teardown scenarios."""
    for a in actors:
        if a is not None:
            try:
                a.destroy()
            except RuntimeError:
                pass

---
## 1. Infrastructure Sensing & World Transformation

### Mathematical Foundation: 2D Image Space to 3D World Space
To send meaningful target tracking coordinates to an Ego vehicle, a static roadside infrastructure camera must map a detected object's pixel coordinate $(u, v)$ back into a 3D World coordinate $(X_w, Y_w, Z_w)$.

This is achieved using the camera Intrinsic Matrix ($K$) and Extrinsic Matrix ($[R|t]$):

$$K = \begin{bmatrix} f_x & 0 & c_x \\ 0 & f_y & c_y \\ 0 & 0 & 1 \end{bmatrix}$$

Given that the ground plane can be structurally approximated ($Z_w \approx 0$), we solve for the scaling factor $\lambda$ to project inverse coordinates from the pixel plane back through the inverted rigid transformation matrix of the camera mount location.

### Configurable Parameters Guide
- **Camera Extrinsics**: Positioning height ($z \ge 6.0\text{m}$) and dramatic downward pitch ($\text{pitch} \approx -35^\circ$) are vital to eliminate self-occlusion artifacts within blind intersections.
- **Intrinsic Matrix Fields**: Tuning Resolution ($W, H$) and Horizontal Field of View ($\text{FOV}$) determines pixel density per meter at long range.

In [3]:
def get_camera_intrinsic_matrix(width, height, fov):
    """Computes the intrinsic matrix K for a pinhole camera model."""
    focal = width / (2.0 * np.tan(fov * np.pi / 360.0))
    K = np.identity(3)
    K[0, 0] = focal
    K[1, 1] = focal
    K[0, 2] = width / 2.0
    K[1, 2] = height / 2.0
    return K

def spawn_roadside_camera(world, transform, width=800, height=600, fov=90, tick=0.05):
    """Spawns a static infrastructure sensor frame."""
    bp = world.get_blueprint_library().find("sensor.camera.rgb")
    bp.set_attribute("image_size_x", str(width))
    bp.set_attribute("image_size_y", str(height))
    bp.set_attribute("fov", str(fov))
    bp.set_attribute("sensor_tick", str(tick))
    return world.spawn_actor(bp, transform)

def image_to_bgr(image):
    """Converts CARLA raw image array to standard BGR layout."""
    arr = np.frombuffer(image.raw_data, dtype=np.uint8)
    arr = np.reshape(arr, (image.height, image.width, 4))
    return arr[:, :, :3].copy()

In [4]:
# ==============================================================================
# CARLA NATIVE SENSOR AND TELEMETRY TEST RUNNER (FIXED SPAWN)
# ==============================================================================

print(">> Cleaning up any leftover vehicles in the world...")
# Automatically find and remove previously spawned vehicles to prevent collision errors
for actor in world.get_actors().filter("vehicle.*.*"):
    try:
        actor.destroy()
        print(f" Removed leftover vehicle ID: {actor.id}")
    except RuntimeError:
        pass

print("\n>> Initializing CARLA API Testing Loop...")

# 1. Access the CARLA World Settings for Synchronous Execution
original_settings = world.get_settings()
settings = world.get_settings()
settings.synchronous_mode = True
settings.fixed_delta_seconds = 0.05  # Fixed time step increments of 50ms
world.apply_settings(settings)

# Track actors spawned natively in this cell
native_test_actors = []

try:
    # 2. Query Blueprint Library for Standard Targets
    tesla_bp = blueprint_library.filter("vehicle.tesla.model3")[0]
    ped_bp = blueprint_library.filter("walker.pedestrian.0001")[0]
    
    # 3. Native Spawning via world.spawn_actor()
    # Shifted x slightly to 118.0 and z to 2.0 to drop cleanly onto the road map surface
    ego_spawn_tf = carla.Transform(carla.Location(x=118.0, y=132.0, z=2.0), carla.Rotation(yaw=0.0))
    test_ego = world.spawn_actor(tesla_bp, ego_spawn_tf)
    native_test_actors.append(test_ego)
    
    # Pedestrian Spawn Location
    ped_spawn_tf = carla.Transform(carla.Location(x=152.0, y=142.0, z=1.5), carla.Rotation(yaw=-90.0))
    test_ped = world.spawn_actor(ped_bp, ped_spawn_tf)
    native_test_actors.append(test_ped)
    
    print(f" Successfully spawned Ego (ID: {test_ego.id}) and Pedestrian (ID: {test_ped.id})")
    
    # 4. Apply Initial Control Vectors
    test_ped.apply_control(carla.WalkerControl(direction=carla.Vector3D(0, -1, 0), speed=1.5))
    test_ego.apply_control(carla.VehicleControl(throttle=0.4, brake=0.0, steer=0.0))
    
    # Tick the world to instantiate entities
    world.tick()
    
    # 5. Core Simulation and Observation Loop (Runs for 100 simulator ticks)
    for frame in range(100):
        # Frame tick signals the server simulator to advance one time step
        world.tick()
        
        # Pull native spatial Transform objects from runtime instances
        ego_transform = test_ego.get_transform()
        ped_transform = test_ped.get_transform()
        
        # Position the Spectator camera perspective procedurally behind the Ego Vehicle
        move_spectator_to(ego_transform, spectator, distance=15.0, z=6.0, pitch=-22.0)
        
        # Calculate straight-line spatial distance using math methods on carla.Location
        distance = ego_transform.location.distance(ped_transform.location)
        
        # Extract precise physical velocity vector using get_velocity()
        velocity_vec = test_ego.get_velocity()
        speed_kmh = 3.6 * math.sqrt(velocity_vec.x**2 + velocity_vec.y**2 + velocity_vec.z**2)
        
        # 6. Apply Native Debug Floating Text to the 3D Render Window
        world.debug.draw_string(
            ego_transform.location + carla.Location(z=2.5), 
            f"SPEED: {speed_kmh:.1f} KM/H", 
            life_time=0.06, 
            color=carla.Color(0, 255, 0)
        )
        
        world.debug.draw_string(
            ped_transform.location + carla.Location(z=2.0), 
            f"DISTANCE TO EGO: {distance:.1f}m", 
            life_time=0.06, 
            color=carla.Color(255, 255, 0)
        )
        
        # Proactive braking check based strictly on local distance measurements
        if distance < 15.0:
            test_ego.apply_control(carla.VehicleControl(throttle=0.0, brake=1.0))
            world.debug.draw_string(
                ego_transform.location + carla.Location(z=3.2), 
                "⚠️ LOCAL BRAKING TRIGGERED", 
                life_time=0.06, 
                color=carla.Color(255, 0, 0)
            )

        if frame % 20 == 0:
            print(f"   [Tick {frame}] Speed={speed_kmh:.1f} km/h | Range={distance:.1f}m")

finally:
    print(">> Cleaning up native testing assets...")
    
    # Clean up actors spawned inside this specific execution block
    for actor in native_test_actors:
        if actor is not None and actor.is_alive:
            actor.destroy()
            
    # CRITICAL: Restore original simulator configurations 
    world.apply_settings(original_settings)
    print(">> Synchronous mode disabled. Environment returned to baseline state.")

>> Cleaning up any leftover vehicles in the world...

>> Initializing CARLA API Testing Loop...
 Successfully spawned Ego (ID: 1472) and Pedestrian (ID: 1473)
   [Tick 0] Speed=3.5 km/h | Range=35.4m
   [Tick 20] Speed=6.2 km/h | Range=35.0m
   [Tick 40] Speed=13.9 km/h | Range=32.3m
   [Tick 60] Speed=18.3 km/h | Range=28.2m
   [Tick 80] Speed=19.4 km/h | Range=23.5m
>> Cleaning up native testing assets...
>> Synchronous mode disabled. Environment returned to baseline state.


In [5]:

# ==============================================================================
# CARLA MULTI-ENVIRONMENT TEST RUNNER (NEW CITY & WEATHER)
# ==============================================================================

print(">> Loading a new environment [Town03: Urban Map with Roundabout]...")
# load_world replaces the current city layout and initializes a fresh world instance
world = client.load_world('Town03') 
blueprint_library = world.get_blueprint_library()
spectator = world.get_spectator()

print(">> Changing weather presets to [Hard Rain Sunset]...")
# Modify environmental lighting, cloud thickness, and active precipitation
world.set_weather(carla.WeatherParameters.HardRainSunset)

# Configure Synchronous Step execution for the newly generated map context
original_settings = world.get_settings()
settings = world.get_settings()
settings.synchronous_mode = True
settings.fixed_delta_seconds = 0.05
world.apply_settings(settings)

# Track clean destruction targets
dynamic_test_actors = []

try:
    # 1. Fetch the map and its standard recommended asset spawn points
    carla_map = world.get_map()
    all_spawn_points = carla_map.get_spawn_points()
    
    if len(all_spawn_points) < 5:
        raise RuntimeError("Insufficient spawn point configurations discovered on this map.")

    # Select two distinctly separated recommended tracking positions
    ego_spawn_tf = all_spawn_points[0]
    ped_spawn_tf = all_spawn_points[2]
    
    # 2. Spawn entities safely using map-native vectors
    tesla_bp = blueprint_library.filter("vehicle.tesla.model3")[0]
    ped_bp = blueprint_library.filter("walker.pedestrian.0001")[0]
    
    print(f" Spawn Point Selected -> Ego: {ego_spawn_tf.location} | Target: {ped_spawn_tf.location}")
    
    test_ego = world.spawn_actor(tesla_bp, ego_spawn_tf)
    dynamic_test_actors.append(test_ego)
    
    test_ped = world.spawn_actor(ped_bp, ped_spawn_tf)
    dynamic_test_actors.append(test_ped)
    
    # 3. Establish initial tracking velocities
    test_ego.apply_control(carla.VehicleControl(throttle=0.35, brake=0.0, steer=0.0))
    test_ped.apply_control(carla.WalkerControl(direction=carla.Vector3D(1, 0, 0), speed=1.4))
    
    # Sync initial state to engine
    world.tick()
    
    print("\n>> Executing Simulation Run under Severe Weather Constraints...")
    for frame in range(120):
        world.tick()
        
        # Pull real-time coordinates
        ego_tf = test_ego.get_transform()
        ped_tf = test_ped.get_transform()
        
        # Position the Spectator tracking camera behind the moving vehicle
        move_spectator_to(ego_tf, spectator, distance=16.0, z=7.0, pitch=-25.0)
        
        # Track distance delta
        distance = ego_tf.location.distance(ped_tf.location)
        
        # Extract precise physical speed
        velocity_vec = test_ego.get_velocity()
        speed_kmh = 3.6 * math.sqrt(velocity_vec.x**2 + velocity_vec.y**2 + velocity_vec.z**2)
        
        # 4. Render tracking data boxes inside the wet/stormy 3D environment viewport
        world.debug.draw_string(
            ego_tf.location + carla.Location(z=2.5), 
            f"WET WEATHER SPEED: {speed_kmh:.1f} KM/H", 
            life_time=0.06, 
            color=carla.Color(0, 255, 255) # Cyan
        )
        
        world.debug.draw_string(
            ped_tf.location + carla.Location(z=2.0), 
            f"DISTANCE: {distance:.1f}m", 
            life_time=0.06, 
            color=carla.Color(255, 100, 0) # Orange
        )
        
        # Implement local reactive safety policy 
        if distance < 18.0: # Increased distance threshold to account for wet roads and sliding
            test_ego.apply_control(carla.VehicleControl(throttle=0.0, brake=1.0))
            world.debug.draw_string(
                ego_tf.location + carla.Location(z=3.2), 
                "⚠️ HAZARD ENCOUNTERED - SLIPPERY BRAKING", 
                life_time=0.06, 
                color=carla.Color(255, 0, 0)
            )
            
        if frame % 20 == 0:
            print(f"   [Frame {frame:03d}] Speed: {speed_kmh:>4.1f} km/h | Target Range: {distance:>5.1f}m")

finally:
    print("\n>> Tearing down dynamic environment configurations...")
    # Clean up spawned elements
    for actor in dynamic_test_actors:
        if actor is not None and actor.is_alive:
            actor.destroy()
            
    # Revert settings 
    world.apply_settings(original_settings)
    print(">> Target tracking system reset completed successfully.")



>> Loading a new environment [Town03: Urban Map with Roundabout]...
>> Changing weather presets to [Hard Rain Sunset]...
 Spawn Point Selected -> Ego: Location(x=-6.446170, y=-79.055023, z=0.275307) | Target: Location(x=76.468864, y=4.308813, z=0.275307)

>> Executing Simulation Run under Severe Weather Constraints...
   [Frame 000] Speed:  3.5 km/h | Target Range: 117.6m
   [Frame 020] Speed:  1.0 km/h | Target Range: 118.4m
   [Frame 040] Speed: 11.7 km/h | Target Range: 117.8m
   [Frame 060] Speed: 16.0 km/h | Target Range: 116.3m
   [Frame 080] Speed: 18.3 km/h | Target Range: 114.3m
   [Frame 100] Speed: 18.2 km/h | Target Range: 112.5m

>> Tearing down dynamic environment configurations...
>> Target tracking system reset completed successfully.


In [ ]:
import carla
import time
import random
import cv2
import queue
import threading
import math
import numpy as np

# ==============================================================================
# 1. CORE SYSTEM UTILITIES AND INITIALIZATION
# ==============================================================================

def move_spectator_to(transform, spectator, distance=12.0, z=6.0, pitch=-25.0):
    """Utility to orient the editor spectator view behind an active actor."""
    back = transform.location - transform.get_forward_vector() * distance
    loc = carla.Location(back.x, back.y, back.z + z)
    rot = carla.Rotation(pitch=pitch, yaw=transform.rotation.yaw, roll=0.0)
    spectator.set_transform(carla.Transform(loc, rot))

def safe_destroy(actors):
    """Safely removes lists of actors under teardown scenarios."""
    for a in actors:
        if a is not None:
            try:
                a.destroy()
            except RuntimeError:
                pass

# Establish connectivity to the running CARLA simulator server
client = carla.Client("localhost", 2000)
client.set_timeout(20.0)

# ==============================================================================
# 2. LOCAL PERCEPTION SIMULATION LOOP (MONITORING CAMERA FEED)
# ==============================================================================

def local_onboard_camera_loop(cam_sensor, stop_event):
    """Simulates an onboard front-facing safety camera for visual validation."""
    frame_queue = queue.Queue(maxsize=1)
    cam_sensor.listen(lambda img: frame_queue.put(img) if not frame_queue.full() else None)
    
    while not stop_event.is_set():
        try:
            image = frame_queue.get(timeout=0.2)
            arr = np.frombuffer(image.raw_data, dtype=np.uint8)
            arr = np.reshape(arr, (image.height, image.width, 4))
            bgr_frame = arr[:, :, :3].copy()
            
            cv2.putText(bgr_frame, "EGO LOCAL ONBOARD CAMERA - UNASSISTED MODE", 
                        (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)
            cv2.imshow("Ego Local View (No V2X)", bgr_frame)
            cv2.waitKey(1)
        except queue.Empty:
            continue
            
    cam_sensor.stop()
    cv2.destroyAllWindows()

# ==============================================================================
# 3. CORE ISOLATED SCENARIO EVALUATION ENGINE
# ==============================================================================

def run_unassisted_scenario(map_name, weather_preset, scenario_id):
    """
    Executes a complete isolated test trial in a specific city/weather configuration
    without any cooperative V2X infrastructure inputs.
    """
    print(f"\n" + "="*70)
    print(f"LAUNCHING SCENARIO {scenario_id}: Map={map_name} | Weather={weather_preset}")
    print("="*70)
    
    # Load requested town configuration
    world = client.load_world(map_name)
    world.set_weather(weather_preset)
    blueprint_library = world.get_blueprint_library()
    spectator = world.get_spectator()
    
    # Enable Synchronous Mode for absolute deterministic execution behavior
    original_settings = world.get_settings()
    settings = world.get_settings()
    settings.synchronous_mode = True
    settings.fixed_delta_seconds = 0.04 # 25 FPS
    world.apply_settings(settings)
    
    trial_actors = []
    stop_signal = threading.Event()
    stop_signal.clear()
    
    # Initialize thread tracker to prevent UnboundLocalError if spawning fails early
    cam_thread = None
    
    # Telemetry indicators
    min_distance_to_target = float("inf")
    collision_detected = False
    speed_at_brake_trigger = 0.0
    brake_timestamp = None
    
    try:
        carla_map = world.get_map()
        spawn_points = carla_map.get_spawn_points()
        
        # 1. SETUP DYNAMIC SPAWNS BASED ON MAP
        if "Town03" in map_name:
            # Explicit intersection arrangement to guarantee an occlusion scenario
            ego_spawn_tf = carla.Transform(carla.Location(x=120.0, y=132.0, z=2.0), carla.Rotation(yaw=0.0))
            vru_spawn_tf = carla.Transform(carla.Location(x=147.0, y=138.0, z=1.5), carla.Rotation(yaw=-90.0))
            vru_direction = carla.Vector3D(0, -1, 0)
        else:
            # Dynamic extraction for alternative maps (like Town05) using native navigation maps
            ego_spawn_tf = spawn_points[0]
            
            # Request a certified walking location from CARLA navigation to guarantee zero collisions
            nav_location = world.get_random_location_from_navigation()
            if nav_location is None:
                # Fallback if navigation mesh isn't loaded completely
                nav_location = spawn_points[min(5, len(spawn_points)-1)].location
                nav_location.z += 1.5
                
            vru_spawn_tf = carla.Transform(nav_location, carla.Rotation(yaw=0.0))
            
            # Point pedestrian toward the vehicle direction dynamically
            vru_direction = (ego_spawn_tf.location - vru_spawn_tf.location).make_unit_vector()

        # 2. SPAWN ACTORS
        tesla_bp = blueprint_library.filter("vehicle.tesla.model3")[0]
        ego_vehicle = world.spawn_actor(tesla_bp, ego_spawn_tf)
        trial_actors.append(ego_vehicle)
        
        ped_bp = blueprint_library.filter("walker.pedestrian.0001")[0]
        vru_actor = world.spawn_actor(ped_bp, vru_spawn_tf)
        trial_actors.append(vru_actor)
        
        # Spawn Onboard Front Verification Camera (Attached to Ego Bumper)
        cam_bp = blueprint_library.find("sensor.camera.rgb")
        cam_bp.set_attribute("image_size_x", "640")
        cam_bp.set_attribute("image_size_y", "480")
        cam_transform = carla.Transform(carla.Location(x=2.0, z=1.3), carla.Rotation(pitch=0.0))
        ego_camera = world.spawn_actor(cam_bp, cam_transform, attach_to=ego_vehicle)
        trial_actors.append(ego_camera)
        
        # 3. RUN PERCEPTION WINDOW THREAD
        cam_thread = threading.Thread(target=local_onboard_camera_loop, args=(ego_camera, stop_signal), daemon=True)
        cam_thread.start()
        
        # Settle physics engine state
        world.tick()
        time.sleep(0.5)
        
        # Direct pedestrian to start walking
        vru_control = carla.WalkerControl(direction=vru_direction, speed=1.5)
        vru_actor.apply_control(vru_control)
        
        # Initial cruise settings
        target_throttle = 0.50
        target_brake = 0.0
        
        start_time = time.time()
        max_duration = 12.0
        
        # 4. EXECUTION SIMULATION LOOP
        while time.time() - start_time < max_duration:
            world.tick()
            
            ego_tf = ego_vehicle.get_transform()
            vru_tf = vru_actor.get_transform()
            move_spectator_to(ego_tf, spectator, distance=15.0, z=6.0, pitch=-22.0)
            
            ego_loc = ego_tf.location
            vru_loc = vru_tf.location
            current_distance = ego_loc.distance(vru_loc)
            
            if current_distance < min_distance_to_target:
                min_distance_to_target = current_distance
                
            if current_distance < 1.95:
                collision_detected = True
                
            # Extract speed telemetry data
            vel = ego_vehicle.get_velocity()
            speed_kmh = 3.6 * math.sqrt(vel.x**2 + vel.y**2 + vel.z**2)
            
            # LOCAL SENSOR PERCEPTION BOUNDS (Line-Of-Sight Emulation Window)
            is_visible_locally = (vru_loc.y - ego_loc.y) < 4.5 and abs(vru_loc.x - ego_loc.x) < 13.0
            
            if is_visible_locally:
                if target_brake == 0.0: # Track exact moment local sensor discovered the danger
                    brake_timestamp = time.time() - start_time
                    speed_at_brake_trigger = speed_kmh
                
                target_throttle = 0.0
                target_brake = 1.0
                world.debug.draw_string(ego_loc + carla.Location(z=2.8), "⚠️ EMERGENCY BRAKE (LOS)", 
                                        life_time=0.05, color=carla.Color(255, 0, 0))
            else:
                world.debug.draw_string(ego_loc + carla.Location(z=2.5), "CRUISE CONTROL ACTIVE", 
                                        life_time=0.05, color=carla.Color(0, 255, 0))
                
            ego_vehicle.apply_control(carla.VehicleControl(throttle=float(target_throttle), brake=float(target_brake), steer=0.0))
            world.debug.draw_string(vru_loc + carla.Location(z=2.0), "WALKER", life_time=0.05, color=carla.Color(0, 255, 255))
            
            time.sleep(0.02)
            
            if target_brake == 1.0 and speed_kmh < 0.1:
                print(">> Ego vehicle reached complete standstill.")
                break
                
    finally:
        # Tear down background operations safely
        stop_signal.set()
        if cam_thread is not None:
            cam_thread.join(timeout=1.5)
        world.apply_settings(original_settings)
        safe_destroy(trial_actors)
        time.sleep(1.0)
        
    return {
        "collision": collision_detected,
        "min_dist": min_distance_to_target,
        "brake_time": brake_timestamp,
        "trigger_speed": speed_at_brake_trigger
    }

# ==============================================================================
# 4. BATCH TRIAL EXECUTION MATRIX AND PROJECT REPORT GENERATOR
# ==============================================================================

# Run Scenario 1: Clear Noon Environment inside Town03
run_1_metrics = run_unassisted_scenario(
    map_name="Town03", 
    weather_preset=carla.WeatherParameters.ClearNoon, 
    scenario_id=1
)

# Run Scenario 2: Hard Rain Sunset environment inside Town05
run_2_metrics = run_unassisted_scenario(
    map_name="Town05", 
    weather_preset=carla.WeatherParameters.HardRainSunset, 
    scenario_id=2
)

# Render Final Quantitative Summary Deliverables Report Table
print("\n" + "="*75)
print("             PROJECT 5 DELIVERABLE: UNASSISTED BASELINE REPORT           ")
print("="*75)
print(f"{'Performance Assessment Metric':<35} | {'Scenario 1 (Town03)':<17} | {'Scenario 2 (Town05)':<15}")
print("-"*75)

outcome_1 = "💥 COLLISION" if run_1_metrics["collision"] else "✅ AVOIDED"
outcome_2 = "💥 COLLISION" if run_2_metrics["collision"] else "✅ AVOIDED"
print(f"{'Safety Validation Outcome':<35} | {outcome_1:<17} | {outcome_2:<15}")

print(f"{'Minimum Spatial Proximity (m)':<35} | {run_1_metrics['min_dist']:<17.2f} | {run_2_metrics['min_dist']:<15.2f}")

time_1 = f"{run_1_metrics['brake_time']:.2f}s" if run_1_metrics['brake_time'] else "N/A"
time_2 = f"{run_2_metrics['brake_time']:.2f}s" if run_2_metrics['brake_time'] else "N/A"
print(f"{'Local Perception Discovery Time':<35} | {time_1:<17} | {time_2:<15}")

speed_1 = f"{run_1_metrics['trigger_speed']:.1f} km/h" if run_1_metrics['brake_time'] else "N/A"
speed_2 = f"{run_2_metrics['trigger_speed']:.1f} km/h" if run_2_metrics['brake_time'] else "N/A"
print(f"{'Speed at Initial Brake Trigger':<35} | {speed_1:<17} | {speed_2:<15}")
print("="*75)
print(">> Project validation script run finished successfully.")


LAUNCHING SCENARIO 1: Map=Town03 | Weather=WeatherParameters(cloudiness=5.000000, precipitation=0.000000, precipitation_deposits=0.000000, wind_intensity=10.000000, sun_azimuth_angle=-1.000000, sun_altitude_angle=45.000000, fog_density=2.000000, fog_distance=0.750000, fog_falloff=0.100000, wetness=0.000000, scattering_intensity=1.000000, mie_scattering_scale=0.030000, rayleigh_scattering_scale=0.033100, dust_storm=0.000000)


RuntimeError: Spawn failed because of collision at spawn position